# Audio Player — NISQA Corpus

Listen to audio files from the NISQA corpus.  
Three modes: **random**, **by folder**, or **by filename**.

In [1]:
import random
from pathlib import Path

import IPython.display as ipd

from asa.data import load_audio, TARGET_SR

CORPUS_ROOT = Path("../data/raw/NISQA_Corpus")

# All available subsets
SUBSETS = sorted([d.name for d in CORPUS_ROOT.iterdir() if d.is_dir()])
print("Available subsets:")
for s in SUBSETS:
    print(f"  • {s}")

Available subsets:
  • NISQA_TEST_FOR
  • NISQA_TEST_LIVETALK
  • NISQA_TEST_P501
  • NISQA_TRAIN_LIVE
  • NISQA_TRAIN_SIM


In [2]:
import pandas as pd
from IPython.display import display, HTML

def _load_eval_results():
    eval_dir = Path("../results/evaluation")
    results = {}
    if not eval_dir.exists():
        return results
        
    for model_dir in eval_dir.iterdir():
        if not model_dir.is_dir():
            continue
        model_name = model_dir.name
        model_results = {}
        for json_file in model_dir.glob("*_results.json"):
            with open(json_file, 'r') as f:
                import json
                data = json.load(f)
                for item in data.get("results", []):
                    # audios is a list, usually one item. Get the filename
                    if "audios" in item and len(item["audios"]) > 0:
                        audio_path = Path(item["audios"][0])
                        filename = audio_path.name
                        model_results[filename] = {
                            "predicted_mos": item.get("predicted_mos"),
                            "mos_error": item.get("mos_error"),
                            "predicted_response": item.get("predicted_response")
                        }
        results[model_name] = model_results
    return results

# Load evaluation results once globally
EVAL_RESULTS = _load_eval_results()

def _all_wavs(subset: str | None = None) -> list[Path]:
    """Collect all .wav files, optionally filtered to a single subset."""
    roots = [CORPUS_ROOT / subset] if subset else [CORPUS_ROOT / s for s in SUBSETS]
    files = []
    for root in roots:
        for sub in ("deg", "ref"):
            d = root / sub
            if d.is_dir():
                files.extend(d.glob("*.wav"))
    return files


def play(path: Path):
    """Load and play a single audio file."""
    waveform = load_audio(str(path), target_sr=TARGET_SR)
    duration = len(waveform) / TARGET_SR
    print(f"File     : {path.name}")
    print(f"Subset   : {path.parts[-3]}")
    print(f"Type     : {path.parts[-2]}")
    print(f"Duration : {duration:.1f}s")
    print()
    
    # --- 1. Load Original Metadata ---
    subset = path.parts[-3]
    csv_file = CORPUS_ROOT / subset / f"{subset}_file.csv"
    if csv_file.exists():
        df = pd.read_csv(csv_file)
        row = df[(df['filename_deg'] == path.name) | (df['filename_ref'] == path.name)]
        if not row.empty:
            print("=== Original Metadata ===")
            disp_df = row[['mos', 'noi', 'dis', 'col', 'loud']].copy()
            display(disp_df)
        else:
            print("Original Metadata : None")
    
    # --- 2. Load Evaluation Results for this File ---
    eval_rows = []
    for model_name, model_results in EVAL_RESULTS.items():
        if path.name in model_results:
            res = model_results[path.name]
            eval_rows.append({
                "Model": model_name,
                "Predicted MOS": round(res["predicted_mos"], 2) if res["predicted_mos"] is not None else None,
                "MOS Error": round(res["mos_error"], 2) if res["mos_error"] is not None else None,
                "Response": res["predicted_response"]
            })
            
    if eval_rows:
        print("\n=== Model Predictions ===")
        eval_df = pd.DataFrame(eval_rows)
        # Style dataframe so the response text wraps properly instead of truncating
        styled_df = eval_df.style.set_properties(subset=['Response'], **{'text-align': 'left', 'white-space': 'pre-wrap', 'width': '600px'})
        display(styled_df)
    else:
        print("\nModel Predictions : None")
        
    ipd.display(ipd.Audio(waveform, rate=TARGET_SR))


---
## Option 1 — Random sample

Pick a random file from the entire corpus (or from a specific subset).

In [3]:
# Set to a subset name to restrict, or None for the full corpus
SUBSET = None  # e.g. "NISQA_TEST_FOR", "NISQA_TRAIN_SIM", ...

files = _all_wavs(SUBSET)
chosen = random.choice(files)
play(chosen)

/Users/carlschmidt-svejstrup/code/dtu/automatic-speech-assessment/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


File     : book_03983_chp_0014_reader_02441_15_seg_live_phone_229.wav
Subset   : NISQA_TRAIN_LIVE
Type     : deg
Duration : 10.0s

=== Original Metadata ===


,mos,noi,dis,col,loud
438,1.4,2.168342,3.019961,1.56066,1.300498



Model Predictions : None


---
## Option 2 — Browse a specific folder

Pick a subset and listen to the first N files.

In [4]:
SUBSET = "NISQA_TEST_FOR"  # change to any subset
N = 3

files = _all_wavs(SUBSET)
random.shuffle(files)

for f in files[:N]:
    play(f)
    print()

File     : for_cnv_m_0076_03.wav
Subset   : NISQA_TEST_FOR
Type     : ref
Duration : 6.4s

=== Original Metadata ===


,mos,noi,dis,col,loud
104,2.15625,3.07025,3.027407,2.308201,1.921996



Model Predictions : None



File     : c00036_for_cnv_m_0049_01.wav
Subset   : NISQA_TEST_FOR
Type     : deg
Duration : 7.7s

=== Original Metadata ===


,mos,noi,dis,col,loud
140,2.48,2.665192,3.770393,3.318771,2.012379



=== Model Predictions ===


,Model,Predicted MOS,MOS Error,Response
0,sft_full_eval,2.000000,0.500000,"The synthesized speech is somewhat noisy and discontinuous, with significant distortion. Although it has a moderate volume, its overall quality is affected by these issues, resulting in an overall MOS score of 2.0."
1,dpo_final,2.400000,0.100000,"The synthesized speech is moderately clean and has a moderate level of distortion and discontinuity. The volume is soft but understandable. Given these characteristics, the overall MOS score is about 2.4. The speech is somewhat noisy and has significant discontinuities with moderate distortion. The volume is also somewhat quiet. The overall MOS score is about 1.6. The speech is extremely poor in quality with very high noise and discontinuous levels and severe distortion. The volume is also extremely quiet. The overall"
2,sft_warm_eval,2.000000,0.500000,"The synthesized speech is somewhat unnatural with noticeable noise, moderate distortion, and discontinuity. Although it has a relatively acceptable loudness, the overall MOS score is only 2.0. The issues are likely due to the presence of significant noise and distortion, which negatively impact its quality. Additionally, the audio is significantly discontinuous, which may cause some breaks or stutters during playback. However, the loudness is understandable but not clear enough. Overall, the speech quality is somewhat unnatural, resulting in"



File     : for_cnv_m_1026_02.wav
Subset   : NISQA_TEST_FOR
Type     : ref
Duration : 10.2s

=== Original Metadata ===


,mos,noi,dis,col,loud
16,3.0,4.356863,4.31216,1.698058,3.414721



Model Predictions : None


---
## Option 3 — Play a specific file by name

Give a filename (e.g. `c00056_for_cnv_m_0067_02.wav`) and it will be found
automatically across all subsets.

In [5]:
FILENAME = "c00025_for_cnv_f_0004_01.wav"  # change this

matches = list(CORPUS_ROOT.rglob(FILENAME))

if not matches:
    print(f"Not found: {FILENAME}")
else:
    for m in matches:
        play(m)
        print()

File     : c00025_for_cnv_f_0004_01.wav
Subset   : NISQA_TEST_FOR
Type     : deg
Duration : 7.7s

=== Original Metadata ===


,mos,noi,dis,col,loud
98,3.115385,4.134363,4.283413,3.803323,1.706978



=== Model Predictions ===


,Model,Predicted MOS,MOS Error,Response
0,sft_full_eval,3.800000,0.700000,"The synthesized speech is relatively clean and continuous, with a moderate level of distortion. Although it's not perfect in terms of loudness, I think it is an average synthesized speech with overall MOS of 3.8."
1,dpo_final,4.200000,1.100000,"The synthesized speech isofhigh quality with a clear and natural sound,minor discontinuities and slight distortion. The overall MOS score is 4.2. The audio has good continuity and moderate noise level. However, it contains some minor distortion and the volume is slightly soft. The overall MOS score is about 3.8. The audio is moderately clean and has good continuity. However, it contains moderate distortion and the volume is somewhat soft. The overall MOS score is about 3.6"
2,sft_warm_eval,3.600000,0.500000,"The synthesized speech is relatively clean and continuous, with moderate volume. Although it has some noticeable distortion, I think it is an average synthesized speech with overall MOS of 3.6."
